# COMP9085 Week 4 Laboratory
## Bayes in healthcare: from prediction to probability

**Two hours.** Week 3 asked whether you could trust the experiment. This lab
asks the Week 4 question: **suppose you trust it, what does the prediction
actually mean?**

You will build a confusion matrix for a real diagnostic model, compute
sensitivity, specificity, PPV and NPV by hand, move the same model into
populations with different prevalence, choose a decision threshold honestly,
and check whether the model's probabilities mean what they say.

**Data.** UCI Breast Cancer Wisconsin (Diagnostic), the same 569 cases and the
same encoding as Weeks 2 and 3: `1 = malignant`. This is **real data used for
classroom evaluation, not evidence of clinical performance.** The prevalence
values in Part B are hypothetical populations chosen for teaching.

**Structure.** The core lab is **Tasks 1–8 and 10–11**. Task 9 is an optional
extension. A checkpoint after Task 8 marks the end of the computational core.

**The Week 4 page is your interactive companion.**

| Tasks | Interactive on the Week 4 page |
|---|---|
| 1 | Confusion-matrix calculator (rows = what is true, columns = what the test says) |
| 2–4 | Diagnostic test (Bayes) calculator and Prevalence explorer |
| 5–7 | Decision threshold explorer |
| 8–9 | Calibration explorer |
| 10 | Which error matters more? (decision reasoning) |
| 11 | The evaluation record |

Explore the calculator and the prevalence explorer before the lab, implement the
same reasoning here in Python, then compare your results with the threshold and
calibration explorers afterwards.

**Deliverable.** This notebook, run top to bottom, with every question answered
in the markdown cells.

### Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (confusion_matrix, roc_auc_score, brier_score_loss, accuracy_score,
                             roc_curve, precision_recall_curve, average_precision_score)

import sklearn
print("scikit-learn", sklearn.__version__, "| numpy", np.__version__)   # for Task 11

SEED = 42
X, y = load_breast_cancer(return_X_y=True)
y = (y == 0).astype(int)        # 1 = malignant, as in Weeks 2 and 3

# The same protocol as Week 3: a stratified development (training) pool and a
# final test set. Every modelling decision (including the threshold) is made on
# the development pool; the final test set is used only after it is frozen.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED)
print(f"train pool {len(y_train)}  |  final test {len(y_test)} "
      f"({y_test.sum()} malignant, {len(y_test) - y_test.sum()} benign)")

---
## Part A — The confusion matrix and the four questions

**Definitions used this week** (identical in the lecture, on the Week 4 page and
here). Rows of the confusion matrix are what is true; columns are what the test
or model says.

| | Formula | Question it answers |
|---|---|---|
| Sensitivity | TP / (TP + FN) | Of those who really have the condition, how many were found? |
| Specificity | TN / (TN + FP) | Of those who do not, how many were cleared? |
| PPV | TP / (TP + FP) | Of those flagged positive, how many really have it? |
| NPV | TN / (TN + FN) | Of those cleared, how many are really clear? |

**Prevalence** is the fraction of the tested population that has the condition.
A case is predicted positive when its probability is at least the threshold:
**positive if p ≥ t** (t = 0.5 unless stated otherwise).

## Task 1 — Build the matrix, and name every cell in biological terms

Fit `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` on the
training pool. Obtain predicted probabilities for the test set with
`predict_proba(X_test)[:, 1]`, classify at the default threshold of 0.5, and
produce the confusion matrix. Print TP, FP, FN and TN separately.

**Question 1.1.** Write one sentence for each cell saying what it means *for a
patient*, e.g. "a false negative is ...".

**Question 1.2.** Suppose, hypothetically, this model were used to decide which
masses are sent for further investigation. Which of the two error cells would
you care about most in that setting, and why? Would your answer change in a
different setting? Is that a statistical judgement or a clinical one?

In [ ]:
# your code here


*Your answer here.*

## Task 2 — Four metrics, four denominators

Using **only** the four counts from Task 1 and plain arithmetic (no sklearn
metric functions), compute sensitivity, specificity, PPV and NPV. For each,
print the fraction as `numerator / denominator = value` so the denominator is
visible.

**Question 2.1.** Sensitivity and PPV both have TP on top. Say, in words, what
group of people each one divides by.

**Question 2.2.** Which two of the four would you expect to stay roughly the
same if this model were used in a hospital where malignancy is much rarer?
Which two would change? (Part B checks your answer.)

In [ ]:
# your code here


*Your answer here.*

---
## Part B — Bayes and the population

## Task 3 — Write Bayes' rule once, and check it against the lecture

Write a function `predictive_values(sens, spec, prev)` that returns
`(ppv, npv)` using Bayes' rule:

$$PPV = \frac{sens \cdot prev}{sens \cdot prev + (1-spec)(1-prev)} \qquad
NPV = \frac{spec (1-prev)}{spec (1-prev) + (1-sens)\, prev}$$

Check it on the lecture's hypothetical example: sensitivity 0.95, specificity
0.95, prevalence 0.01. You should get PPV ≈ 0.161 and NPV ≈ 0.9995.

**Question 3.1.** Redo the same example as **natural frequencies** for 10,000
people: how many are ill, how many test positive, how many of the positives are
ill? Which representation would you use to explain the result to a patient?

In [ ]:
# your code here


*Your answer here.*

## Task 4 — Same model, different populations

Take the **sensitivity and specificity your model measured in Task 2** and use
`predictive_values` to compute PPV and NPV at four hypothetical prevalences:
the test set's own prevalence, 0.10, 0.01 and 0.001. Put the results in a
table. Then plot PPV and NPV against prevalence from 0.001 to 0.5 on a
log-scaled x-axis.

**Question 4.1.** The model did not change between the rows of your table.
What did change, and why does it lower the PPV so much at 0.001? What could be
changed in practice to raise it?

**Question 4.2.** WDBC is about 37 per cent malignant. What does that tell you
about quoting the PPV you measured on this test set to a population screening
programme?

**Question 4.3.** Open the **Diagnostic test (Bayes) calculator** on the
Week 4 page, set sensitivity and specificity to your measured values and the
prevalence to 1%. Does the calculator's PPV agree with your function? Then use
the **Prevalence explorer** to see the same effect across five populations at
once.

In [ ]:
# your code here


*Your answer here.*

---
## Part C — The decision threshold

## Task 5 — Sweep the threshold

On the **training pool only**, obtain out-of-fold probabilities with
`cross_val_predict(lr, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]`.
For thresholds from 0.05 to 0.95 in steps of 0.05 compute TP, FP, FN, TN,
sensitivity and specificity. Show the table and plot sensitivity and
specificity against the threshold.

**The protocol for Part C (write it down; Week 3 applies):**

> out-of-fold (OOF) development predictions → threshold selection → **freeze the
> threshold** → apply it to the final test set **once**

Out-of-fold (OOF) predictions are cross-validated predictions: each training case
is scored by a model that did not see it during fitting.

**Question 5.1.** Describe in one sentence what happens to each error type as
the threshold falls.

**Question 5.2.** Why are we using cross-validated predictions on the training
pool here rather than the test set? (Week 3 vocabulary, please.)

In [ ]:
# your code here


*Your answer here.*

## Task 6 — ROC and precision–recall curves

Using the same **out-of-fold** probabilities `p_oof` from Task 5, plot the ROC
curve (`roc_curve`) with its AUC, and the precision–recall curve
(`precision_recall_curve`) with its average precision. Mark the point that
corresponds to the 0.5 threshold on both.

The precision–recall (PR) curve plots precision (PPV) against recall
(sensitivity). Because precision depends on how many positives there are, the PR
curve is particularly informative when positives are rare: it shows the
false-alarm burden that a ROC curve can make look small.

**Question 6.1.** What does the AUC summarise, and what two things does it NOT
tell you? (Hint: one is a decision, the other is Part D.)

**Question 6.2.** Precision is PPV. Which of the two curves would change if the
same model were used in a population with 1% prevalence, and why? Which curve
would you show for a rare condition?

In [ ]:
# your code here


*Your answer here.*

## Task 7 — Choose a threshold for a stated purpose, then test it once

**Hypothetical scenario.** Suppose the model were used to decide which masses get
**further investigation**, and a (hypothetical) clinical team asked for
sensitivity of at least 0.97. Using the **out-of-fold** table from Task 5,
choose the highest threshold that meets that target and **freeze it**. Only then
apply it to the final test set, once, and report the confusion matrix,
sensitivity, specificity, PPV and NPV.

**Question 7.1.** Compared with the 0.5 threshold in Task 1, what did you gain
and what did you pay?

**Question 7.2.** Did the test set meet the 0.97 target exactly? If not, why
not, and what does that say about reporting a single test-set number?

**Question 7.3.** Describe one concrete setting where this threshold would be
used. What does a false positive cost there, what does a false negative cost,
and whose interests did you weight? This is a judgement, not a calculation:
say so.

In [ ]:
# your code here


*Your answer here.*

---
## Part D — Is 0.8 really 80%? Calibration

## Task 8 — Two models, similar accuracy, different probabilities

Fit a second model, `GaussianNB()`, on the training pool. (Naive Bayes is next
week's classifier: consider this a preview.)

Keep development and final testing separate:

* **Development (calibration analysis):** draw a **reliability diagram** for each
  model using `calibration_curve(y, p, n_bins=5)` on **out-of-fold predictions
  from the training pool** (more cases than the test set, and still honest), with
  the diagonal for reference. Add a histogram of each model's predicted
  probabilities.
* **Final test set (report once):** for both models, report accuracy at 0.5, AUC
  and Brier score. The Brier score is the mean squared difference between the
  predicted probability and the outcome (0 or 1); lower is better.

Calibration is assessed over groups of predictions: a bin of cases predicted
near 0.8 should be about 80% positive. It does not tell you the outcome for any
one case.

**Question 8.1.** Which model's probabilities would you trust as
probabilities? Use the diagram, not the accuracy.

**Question 8.2.** What fraction of Gaussian naive Bayes predictions are below
0.05 or above 0.95? Why does a model that is that sure of itself worry you in a
clinical setting?

**Question 8.3.** Is logistic regression well calibrated here? Point to the
bins that support your answer, and say which bins you trust least and why.

**Question 8.4.** The two models have almost the same AUC but different Brier
scores. Explain the difference between discrimination and calibration using
this result.

*The Calibration explorer on the Week 4 page separates calibration from AUC with a slider.*

In [ ]:
# your code here


*Your answer here.*

---
## ✅ CORE LAB COMPLETE: computational core (Tasks 1–8)

You have completed every computational task in the core lab. **Task 9 is an
optional extension.** Tasks 10 and 11 are short written tasks that complete the
core lab: do them even if you skip Task 9.

## Task 9 (optional extension) — Can calibration be repaired?

Wrap Gaussian naive Bayes in `CalibratedClassifierCV(GaussianNB(), method="sigmoid", cv=5)`,
fit it on the training pool and compare its test-set Brier score and AUC with
the uncalibrated model.

**Question 9.1.** Which number improved, and which barely moved? Explain why
recalibration cannot turn a model that ranks badly into one that ranks well.

In [ ]:
# your code here


*Your answer here.*

---
## Part E — Which metric matters depends on the question

## Task 10 — Three questions, three answers

No code. For each scenario, state which metric(s) you would prioritise, roughly
where you would put the threshold (low / default / high), what prevalence you
would assume, and what you would need to know that this notebook cannot tell
you.

1. **Screening:** deciding which people in a general population are invited for
   a cheap, safe follow-up scan.
2. **Confirmation:** deciding which patients proceed to surgery.
3. **Discovery:** ranking 2,000 candidate biomarkers so that the top 20 can be
   validated with an expensive laboratory assay.

**Question 10.1.** Is there one "best" metric across all three? Explain.

*Your answer here.*

## Task 11 — Write the evaluation record

Week 3 ended with a reproducibility record. Complete this one for your Task 7
result, in a markdown table: dataset and version; split protocol and seed;
model; how the threshold was chosen and on which data; the test-set confusion
matrix; sensitivity, specificity, PPV and NPV with their denominators; the
prevalence at which PPV and NPV were measured; a calibration statement; and one
sentence on what this result does **not** show.

**Question 11.1.** Why must the prevalence appear in the record whenever PPV or
NPV does?

In [ ]:
# Versions for the record
import sklearn, sys
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__,
      "| numpy", np.__version__, "| pandas", pd.__version__)

*Your answer here.*